Hier wird analysiert und dargestellt das Einbeziehen welcher features denn wie viele samples liefert.

In [4]:
import pandas as pd
import numpy as np

# =========================================================
# 1. Daten laden & bereinigen (wie im Haupt-Skript)
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"
df = pd.read_csv(file_path, sep=";", decimal=",")

df['station_id'] = df['station_id'].astype(str).str.strip()

# Kaputte / verzerrende Sensoren vorab entfernen
bad_sensors = ['5456', '15449', '46505', '9094']
df = df[~df['station_id'].isin(bad_sensors)].copy()

# =========================================================
# 2. Szenarien (Feature-Sets) definieren
# =========================================================
scenarios = [
    {
        "Name": "Scenario 1 (Only ICON-D2 & ERA5 data)",
        "Features": ['icon_temp_2m', 'icon_wind_speed', 'icon_wind_direction', 'ERA5_BLH']
    },
    {
        "Name": "Scenario 2 (only traffic)",
        "Features": ['IDW_car', 'IDW_truck', 'prox_kumulativ_car', 'prox_kumulativ_truck']
    },
    {
        "Name": "Scenario 3 (only Sentinel-5P data)",
        "Features": ['S5P_NO2', 'S5P_UVAI']
    },
    {
        "Name": "Szenario 4 (only MODIS data)",
        "Features": [ 'MAIAC_AOD']
    },
    {
        "Name": "Szenario 5 (all Features)",
        "Features": ['icon_temp_2m', 'S5P_NO2', 'S5P_UVAI', 'ERA5_BLH',
                     'IDW_car', 'IDW_truck', 'prox_kumulativ_car', 'prox_kumulativ_truck',
                     'icon_wind_speed', 'icon_wind_direction',
                     'ndvi', 'tpi_300m', 'tpi_1500m', 'icon_precip_12h', "MAIAC_AOD"]
    }
]

# =========================================================
# 3. Schleife: Sample Size berechnen
# =========================================================
print("Berechne verfügbare Samples pro Parameter-Szenario...\n")

results = []
target = 'PM25_Target'

for sc in scenarios:
    # Wir droppen alle Zeilen, die im Target ODER in den aktuellen Features ein NaN haben
    cols_to_check = [target] + sc["Features"]
    df_filtered = df.dropna(subset=cols_to_check)

    # Speichern der Ergebnisse
    results.append({
        "Szenario": sc["Name"],
        "Verwendete Parameter": ", ".join(sc["Features"]),
        "Verfügbare Samples": len(df_filtered)
    })

# =========================================================
# 4. Als saubere Tabelle ausgeben
# =========================================================
results_df = pd.DataFrame(results)

# Optional: Speichere die Tabelle für deine Thesis ab
output_csv = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Diagrams\Sample_Size_Scenarios.csv"
results_df.to_csv(output_csv, sep=";", index=False, encoding='utf-8-sig')

# Tabelle in der Konsole anzeigen
pd.set_option('display.max_colwidth', None) # Damit die Parameter-Spalte nicht abgeschnitten wird
print(results_df.to_string(index=False))
print(f"\n-> Die Übersicht wurde auch hier gespeichert: {output_csv}")

Berechne verfügbare Samples pro Parameter-Szenario...

                             Szenario                                                                                                                                                                                 Verwendete Parameter  Verfügbare Samples
Scenario 1 (Only ICON-D2 & ERA5 data)                                                                                                                                         icon_temp_2m, icon_wind_speed, icon_wind_direction, ERA5_BLH                 572
            Scenario 2 (only traffic)                                                                                                                                         IDW_car, IDW_truck, prox_kumulativ_car, prox_kumulativ_truck                 572
   Scenario 3 (only Sentinel-5P data)                                                                                                                                               

In [5]:

# =========================================================
# Analyse der Datenverfügbarkeit: MAIAC AOD & Sentinel-5P
# =========================================================

target = "PM25_Target"

# Nur Zeilen berücksichtigen, für die der Zielwert vorhanden ist,
# da sie grundsätzlich für ein RF-Modell infrage kommen.
df_target = df.dropna(subset=[target]).copy()

n_total = len(df)
n_target = len(df_target)

print("=== Datengrundlage ===")
print(f"Beobachtungen nach Sensorbereinigung: {n_total}")
print(f"Beobachtungen mit vorhandenem PM2.5-Target: {n_target}")
print()

# ---------------------------------------------------------
# 1. Fehlende Daten pro Feature
# ---------------------------------------------------------

features_to_check = ["MAIAC_AOD", "S5P_NO2", "S5P_UVAI"]

results = []

for col in features_to_check:
    missing = df_target[col].isna().sum()
    available = df_target[col].notna().sum()

    results.append({
        "Feature": col,
        "Fehlende Werte": missing,
        "Verfügbare Werte": available,
        "Fehlend (%)": round(missing / n_target * 100, 2)
            if n_target > 0 else np.nan
    })

missing_df = pd.DataFrame(results)

print("=== Fehlende Daten pro Feature ===")
print(missing_df.to_string(index=False))
print()

# ---------------------------------------------------------
# 2. Datenverfügbarkeit von Sentinel-5P als Datenquelle
# ---------------------------------------------------------

s5p_cols = ["S5P_NO2", "S5P_UVAI"]

# Für ein Modell mit beiden Sentinel-5P-Features gilt eine Zeile
# als unvollständig, sobald mindestens eines der Features fehlt.
s5p_missing_any = df_target[s5p_cols].isna().any(axis=1)

# Beide Sentinel-5P-Features fehlen gleichzeitig.
s5p_missing_both = df_target[s5p_cols].isna().all(axis=1)

print("=== Sentinel-5P insgesamt ===")
print(
    f"Mindestens ein Sentinel-5P-Feature fehlt: "
    f"{s5p_missing_any.sum()} "
    f"({s5p_missing_any.mean() * 100:.2f} %)"
)
print(
    f"Beide Sentinel-5P-Features fehlen: "
    f"{s5p_missing_both.sum()} "
    f"({s5p_missing_both.mean() * 100:.2f} %)"
)
print()

# ---------------------------------------------------------
# 3. Wie stark beschränken MAIAC AOD und Sentinel-5P
#    gemeinsam die verfügbare Sample Size?
# ---------------------------------------------------------

source_cols = ["MAIAC_AOD", "S5P_NO2", "S5P_UVAI"]

# Zeilen, bei denen mindestens eine dieser Variablen fehlt
source_missing_any = df_target[source_cols].isna().any(axis=1)

# Zeilen, bei denen alle drei Variablen verfügbar sind
source_available_all = df_target[source_cols].notna().all(axis=1)

n_excluded = source_missing_any.sum()
n_available = source_available_all.sum()

pct_excluded = (
    n_excluded / n_target * 100 if n_target > 0 else np.nan
)
pct_available = (
    n_available / n_target * 100 if n_target > 0 else np.nan
)

print("=== Gemeinsamer Einfluss auf die Sample Size ===")
print(
    f"Samples mit fehlendem MAIAC- oder Sentinel-5P-Wert: "
    f"{n_excluded} ({pct_excluded:.2f} %)"
)
print(
    f"Samples mit allen drei Features verfügbar: "
    f"{n_available} ({pct_available:.2f} %)"
)
print(
    f"Ausgangsbasis mit vorhandenem PM2.5-Target: {n_target}"
)


=== Datengrundlage ===
Beobachtungen nach Sensorbereinigung: 572
Beobachtungen mit vorhandenem PM2.5-Target: 572

=== Fehlende Daten pro Feature ===
  Feature  Fehlende Werte  Verfügbare Werte  Fehlend (%)
MAIAC_AOD             403               169        70.45
  S5P_NO2             298               274        52.10
 S5P_UVAI               0               572         0.00

=== Sentinel-5P insgesamt ===
Mindestens ein Sentinel-5P-Feature fehlt: 298 (52.10 %)
Beide Sentinel-5P-Features fehlen: 0 (0.00 %)

=== Gemeinsamer Einfluss auf die Sample Size ===
Samples mit fehlendem MAIAC- oder Sentinel-5P-Wert: 427 (74.65 %)
Samples mit allen drei Features verfügbar: 145 (25.35 %)
Ausgangsbasis mit vorhandenem PM2.5-Target: 572
